# LangChain Agent with a Self-Hosted LLM on Azure Container Apps

This notebook demonstrates how to build an **AI agent** using **LangChain** and **LangGraph** that connects to a self-hosted Gemma 4 model running on Azure Container Apps (ACA) with vLLM.

Since vLLM exposes an **OpenAI-compatible API**, we use LangChain's `ChatOpenAI` class to connect — no Azure OpenAI resource required.

## Key Concepts

1. **OpenAI** — The OpenAI-compatible API spec that vLLM implements, which LangChain can interface with.
2. **ChatOpenAI** — LangChain's chat model wrapper, pointed at our custom vLLM endpoint.
3. **Tools** — Python functions decorated with `@tool` that the agent can invoke.
4. **ReAct Agent** — A LangGraph prebuilt agent that reasons, calls tools, and synthesizes answers.

## Deploying the infrastructure on Azure

To deploy the required infrastructure on Azure, we use `Terraform`. The Terraform code provisions:
- An Azure Container Apps environment
- A serverless workload profile that uses GPU-enabled instances
- A container app running vLLM with the Gemma 4 31B IT model
- A session pool for Python REPL tool
- A container app for hosting MCP server for searching the web

Deploy these resources using the following `terraform` commands in the `555_llm_on_aca_gpu` directory:

In [ ]:
%terraform init
%terraform apply -auto-approve

## Get the LLM Endpoint

To consume the self-hosted model, we need to provide the base URL of our vLLM deployment and an API key (which can be any non-empty string since we disabled authentication in vLLM).

Retrieve the FQDN of the Gemma 4 model deployed on ACA from the Terraform output.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


Before building our agent, let's start with the basics. To consume an LLM model, OpenAI defines a standard API spec. It uses `openai` object with methods like `openai.chat.completions.create()`. vLLM implements this same API, so we can use it to connect to our self-hosted model.
Let's first install the OpenAI Python SDK, which we will use to call our vLLM model.

In [21]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Now we can consume the self-hosted model using OpenAI API.

In [19]:
from openai import OpenAI

client = OpenAI(
    # base_url=f"http://4.165.31.28/v1",
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY"
)

response = client.chat.completions.create(
    model="google/gemma-4-31B-it",
    messages=[
        {"role": "user", "content": "What is Azure Container Apps?"}
    ],
    max_tokens=512,
    temperature=1.0,
    stream=True
)

for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft. It is designed for developers who want to build and deploy modern, microservices-based applications without having to manage the underlying infrastructure (like virtual machines or Kubernetes clusters).

Think of it as the "middle ground" between **Azure App Service** (simple, single-container apps) and **Azure Kubernetes Service (AKS)** (full control, complex orchestration).

Here is a detailed breakdown of what makes Azure Container Apps unique:

---

### 1. The Core Philosophy: "Serverless Kubernetes"
Under the hood, Azure Container Apps runs on **Azure Kubernetes Service (AKS)** and **KEDA** (Kubernetes-based Event Driven Autoscaling). However, the "Kubernetes" part is completely hidden from you. You don’t manage nodes, clusters, or control planes; you simply provide the container image, and Azure handles the rest.

### 2. Key Features

#### **Automatic Scaling (KEDA)**
One of the

Using `openai` API is enough to call the model, but it does not provide a great developer experience. So frameworks like `LangChain`, `Microsoft Agent Framework`, `Crew`, etc, can help to provide more rich features like memory management, conversation handling, tool integration and creating agents.

Next, we will use LangChain's `ChatOpenAI` class, which provides a more convenient interface and additional features for working with chat models.

In [26]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Simple Chat — No Tools

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the vLLM OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output. And note also how much similar the code is to calling OpenAI's API directly.

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 4096 # 131072, # 512
)

response = model.stream([HumanMessage(content="What is Azure Container Apps?")])

for chunk in response:
    print(chunk.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft. It allows you to deploy and scale applications and microservices without having to manage the underlying infrastructure (like virtual machines or Kubernetes clusters).

Think of it as a middle ground: it gives you the **power of Kubernetes** (orchestration, scaling, networking) but with the **simplicity of Serverless** (no cluster management, pay-as-you-go).

Here is a detailed breakdown of what it is and why it matters.

---

### 1. The Core Concept: "Serverless Kubernetes"
Under the hood, Azure Container Apps is built on **Azure Kubernetes Service (AKS)** and **KEDA** (Kubernetes Event-driven Autoscaling), but all the "Kubernetes complexity" is hidden from you.

*   **In AKS:** You have to manage nodes, upgrade versions, configure ingress controllers, and manage pods.
*   **In ACA:** You simply provide a container image (from Docker Hub or Azure Container Registry), and Azure handle

## 2. Define Tools

Create custom Python functions as tools using the `@tool` decorator. These will be available for the agent to call when needed.

In [4]:
from langchain_core.tools import tool
from random import randint


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {temp}°C."


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.

    Args:
        a: first number
        b: second number
    """
    return a * b


tools = [get_weather, multiply]
print("Registered tools:", [t.name for t in tools])

Registered tools: ['get_weather', 'multiply']


## 3. Tool Binding — Test Tool Calling

Bind the tools to the model and verify the LLM can decide when to call them.

In [5]:
model_with_tools = model.bind_tools(tools)

# This should NOT trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="Hi there!")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: Hello! How can I help you today?
Tool calls: []


In [6]:
# This SHOULD trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="What's the weather in Amsterdam?")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: 
Tool calls: [{'name': 'get_weather', 'args': {'location': 'Amsterdam'}, 'id': 'chatcmpl-tool-986a9d83570a7c35', 'type': 'tool_call'}]


## 4. Create a ReAct Agent

Use LangGraph's `create_react_agent` to build an agent that can reason about when to call tools, execute them, and incorporate results into its response.

The agent implements the **ReAct** (Reasoning + Acting) pattern: it thinks about what to do, calls a tool if needed, observes the result, and repeats until it has a final answer.

In [7]:
from langchain.agents import create_agent

agent = create_agent(model, tools)

## 5. Run the Agent

### No tool needed — simple question

In [8]:
response = agent.invoke({"messages": [HumanMessage(content="Hi! What is Kubernetes?")]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! What is Kubernetes?
================================== Ai Message ==================================

**Kubernetes** (often abbreviated as **K8s**) is an open-source platform designed to automate the deploying, scaling, and operating of application containers.

To understand Kubernetes, it helps to understand the concepts it builds upon:

### 1. The Background: Containers
Before Kubernetes, we had containers (like **Docker**). A container packages an application together with everything it needs to run (libraries, dependencies, configuration). This ensures the app runs the same way whether it's on a developer's laptop or a production server.

However, if you have hundreds or thousands of containers running across dozens of different servers, managing them manually becomes impossible. You have to worry about:
*   Which server has enough RAM to hold a new container?
*   What happens if a server crashes?

### Tool call — weather query

The agent should recognize this requires the `get_weather` tool, call it, then respond with the result.

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather like in Amsterdam?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the weather like in Amsterdam?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-af58a8047b2cdb82)
 Call ID: chatcmpl-tool-af58a8047b2cdb82
  Args:
    location: Amsterdam
================================= Tool Message =================================
Name: get_weather

The weather in Amsterdam is cloudy with a high of 10°C.
================================== Ai Message ==================================

The weather in Amsterdam is currently cloudy with a high of 10°C.


### Tool call — multiply

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 7 multiplied by 13?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 7 multiplied by 13?
================================== Ai Message ==================================
Tool Calls:
  multiply (chatcmpl-tool-8d644489737ca673)
 Call ID: chatcmpl-tool-8d644489737ca673
  Args:
    a: 7
    b: 13
================================= Tool Message =================================
Name: multiply

91
================================== Ai Message ==================================

7 multiplied by 13 is 91.


## 6. Streaming

Stream the agent's step-by-step reasoning and responses in real time. Each step (LLM thinking, tool call, tool result, final answer) is printed as it occurs.

In [11]:
for step in agent.stream(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather in Paris?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-9743055666736ef1)
 Call ID: chatcmpl-tool-9743055666736ef1
  Args:
    location: Paris
================================= Tool Message =================================
Name: get_weather

The weather in Paris is sunny with a high of 12°C.
================================== Ai Message ==================================

The weather in Paris is sunny with a high of 12°C.


## 7. Microsoft Learn MCP Server

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets you expose tools via a standard protocol that any MCP-compatible client can consume. This is especially powerful for **remote hosted MCP servers** — tools running as HTTP services that your agent can call over the network.

Microsoft exposes a public MCP server at `https://learn.microsoft.com/api/mcp` that provides tools to search and retrieve Microsoft documentation.

This is a great example of a **third-party hosted MCP server** — you don't need to deploy anything, just point your client at the URL.

In [12]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the Microsoft Learn MCP server
learn_client = MultiServerMCPClient(
    {
        "microsoft-learn": {
            "url": "https://learn.microsoft.com/api/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by Microsoft Learn
learn_tools = await learn_client.get_tools()
print("Microsoft Learn MCP tools:", [t.name for t in learn_tools])

# Create an agent with the self-hosted LLM + Microsoft Learn tools
learn_agent = create_agent(model, learn_tools)

Microsoft Learn MCP tools: ['microsoft_docs_search', 'microsoft_code_sample_search', 'microsoft_docs_fetch']


### Query Microsoft Documentation

Ask the agent a question that requires looking up Microsoft documentation. The agent will call the Microsoft Learn MCP tools to search and retrieve relevant docs.

In [13]:
from langchain_core.messages import HumanMessage

async for step in learn_agent.astream(
    {"messages": [HumanMessage(content="How do I deploy a container app with GPU support on Azure Container Apps?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How do I deploy a container app with GPU support on Azure Container Apps?
================================== Ai Message ==================================
Tool Calls:
  microsoft_docs_search (chatcmpl-tool-ace4d8cb312b235f)
 Call ID: chatcmpl-tool-ace4d8cb312b235f
  Args:
    query: Azure Container Apps GPU support deployment guide
================================= Tool Message =================================
Name: microsoft_docs_search

[{'type': 'text', 'text': '{"results":[{"title":"Tutorial: Deploy AI image generation with serverless GPUs (azure-portal)","content":"# Tutorial: Deploy AI image generation with serverless GPUs (azure-portal)\\n## Deploy by using the Azure portal\\nFollow these steps to create a GPU-enabled container app and deploy the image generation solution by using the Azure portal.\\n### Create a Container Apps environment with GPU\\n1. In the [Azure portal](https://portal.azure.c

## 8. Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

In [14]:
aca_mcp_server_fqdn = ! terraform output -raw aca_mcp_server_fqdn
aca_mcp_server_fqdn = aca_mcp_server_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_fqdn)

MCP Server Endpoint: mcp-server.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [15]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [16]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools

In [ ]:
# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor],
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
# model.max_completion_tokens=131072
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [30]:
from langchain_core.messages import HumanMessage

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""Search the web for the latest news about Azure Container Apps in 2026.
                                          Limit search to 2 results only.
                                          Fetch and analyse each web page one by one.
                                          Use official sources only.""")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Search the web for the latest news about Azure Container Apps in 2026.
                                          Limit search to 2 results only.
                                          Fetch and analyse each web page one by one.
                                          Use official sources only.
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-9d38c7f3a6e24b12)
 Call ID: chatcmpl-tool-9d38c7f3a6e24b12
  Args:
    limit: 2
    query: site:microsoft.com "Azure Container Apps" 2026
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "site:microsoft.com \\"Azure Container Apps\\" 2026",\n  "engines": [\n    "duckduckgo"\n  ],\n  "totalResults": 2,\n  "results": [\n    {\n      "title": "Azure Container Apps overview | Microsoft Learn",\n      "url": "https:

### Streaming with MCP Tools

Stream the agent's step-by-step reasoning as it decides to call the remote MCP web search tool and synthesizes the results.

In [ ]:
async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="What is the current price of Bitcoin?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current price of Bitcoin?
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-bd6beb9d43c83e5c)
 Call ID: chatcmpl-tool-bd6beb9d43c83e5c
  Args:
    query: current price of Bitcoin
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "current price of Bitcoin",\n  "engines": [\n    "duckduckgo"\n  ],\n  "totalResults": 10,\n  "results": [\n    {\n      "title": "Bitcoin price today, BTC to USD live price, marketcap and chart ...",\n      "url": "https://coinmarketcap.com/currencies/bitcoin/",\n      "description": "The live <b>Bitcoin</b> <b>price</b> today is $68,153.75 USD with a 24-hour trading volume of $34,210,076,040.38 USD. We update our BTC to USD <b>price</b> in real-time.",\n      "source": "coinmarketcap.com",\n      "engine": "duckduckg

In [29]:
sessionpool_management_endpoint = ! terraform output -raw sessionpool_management_endpoint
sessionpool_management_endpoint = sessionpool_management_endpoint.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint)

mcp_sessionpool_endpoint = ! terraform output -raw mcp_sessionpool_endpoint
mcp_sessionpool_endpoint = mcp_sessionpool_endpoint.n
print("MCP Session Pool Endpoint:", mcp_sessionpool_endpoint)

Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool/mcp


In [ ]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ?
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (chatcmpl-tool-83eb2de34cbec3f2)
 Call ID: chatcmpl-tool-83eb2de34cbec3f2
  Args:
    python_code: from datetime import datetime
import pytz

tunisia_tz = pytz.timezone('Africa/Tunis')
france_tz = pytz.timezone('Europe/Paris')

tunisia_time = datetime.now(tunisia_tz).strftime('%H:%M:%S')
france_time = datetime.now(france_tz).strftime('%H:%M:%S')

print(f"Tunisia: {tunisia_time}")
print(f"France: {france_time}")
================================= Tool Message =================================
Name: Python_REPL

{
  "result": "",
  "stdout": "Tunisia: 20:07:01\nFrance: 21:07:01\n",
  "stderr": ""
}
================================== Ai Message ==================================

The current time is:
* **Tunisia:** 20:07 (8:07 PM)
* **France:** 21:07 (9:07

The tool supports file operations as well:

In [ ]:
# from langchain_azure_dynamic_sessions import SessionsBashTool

# tool = SessionsBashTool(pool_management_endpoint=sessionpool_management_endpoint, access_token_provider=access_token_provider)

# response = tool.execute("echo Hello from the session!")

# # response

# # # upload a file to the session
# # tool.upload_file(local_file_path="./rg.tf", remote_file_path="/mnt/user/rg.tf")

# # # list files in the session
# # files = tool.list_files()

# # # download a file from the session
# # tool.download_file(remote_file_path="/mnt/user/output.txt", local_file_path="./output.txt")

In [ ]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

In [31]:
tools = [get_weather, multiply, toolPythonSession]
print("Registered tools:", [t.name for t in tools])

Registered tools: ['get_weather', 'multiply', 'Python_REPL']


## More Resources

- [LangChain Tool Calling](https://python.langchain.com/docs/concepts/tool_calling/)
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) — the open standard for tool interoperability
- [Microsoft Learn MCP Server](https://learn.microsoft.com/api/mcp) — search Microsoft documentation via MCP